# 03 — Infraestrutura Digital e Gap Bancário

**Objetivo**: avaliar conectividade, infraestrutura digital e concorrência física dos bancos tradicionais.

**Inputs**: `data/processed/trusted_municipios_eda.parquet`.

**Outputs**: figuras de infraestrutura, gap bancário e matriz de quadrantes.

In [ ]:
# Imports
import logging

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.config import PROCESSED_DATA_DIR
from src.utils.eda import (
    plot_boxplot_by_group,
    plot_distribution,
    save_figure,
    save_json,
)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# Leitura
df = pd.read_parquet(PROCESSED_DATA_DIR / "trusted_municipios_eda.parquet")
logger.info("Linhas: %d, Colunas: %d", df.shape[0], df.shape[1])

## 1. Municípios sem agência bancária

In [ ]:
sem_agencia = df["flag_sem_agencia"].sum()
logger.info(
    "Municípios sem agência: %d (%.2f%%)",
    sem_agencia,
    sem_agencia / len(df) * 100,
)

## 2. Distribuição de infraestrutura digital e gap bancário

In [ ]:
plot_distribution(
    df, "banda_larga_fixa_por_100_hab", filename="03_dist_banda_larga.png"
)
plt.show()

plot_distribution(
    df, "agencias_por_100k_hab", filename="03_dist_agencias.png"
)
plt.show()

## 3. Boxplots por região

In [ ]:
for col in ["banda_larga_fixa_por_100_hab", "agencias_por_100k_hab", "depositos_per_capita"]:
    plot_boxplot_by_group(df, col, "nome_regiao", filename=f"03_boxplot_{col}_regiao.png")
    plt.show()

## 4. Quadrantes: infraestrutura digital × gap bancário

In [ ]:
mediana_banda = df["banda_larga_fixa_por_100_hab"].median()
mediana_agencias = df["agencias_por_100k_hab"].fillna(0).median()

df["quadrante"] = "desconectado"
df.loc[
    (df["banda_larga_fixa_por_100_hab"] >= mediana_banda)
    & (df["agencias_por_100k_hab"].fillna(0) < mediana_agencias),
    "quadrante",
] = "alto_potencial"
df.loc[
    (df["banda_larga_fixa_por_100_hab"] >= mediana_banda)
    & (df["agencias_por_100k_hab"].fillna(0) >= mediana_agencias),
    "quadrante",
] = "maduro_saturado"
df.loc[
    (df["banda_larga_fixa_por_100_hab"] < mediana_banda)
    & (df["agencias_por_100k_hab"].fillna(0) >= mediana_agencias),
    "quadrante",
] = "bancarizado_sem_infra"

logger.info("Distribuição dos quadrantes:\n%s", df["quadrante"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.scatterplot(
    data=df,
    x="banda_larga_fixa_por_100_hab",
    y="agencias_por_100k_hab",
    hue="quadrante",
    alpha=0.7,
    ax=ax,
)
ax.axvline(mediana_banda, color="gray", linestyle="--")
ax.axhline(mediana_agencias, color="gray", linestyle="--")
ax.set_title("Infraestrutura Digital vs Gap Bancário")
ax.set_xlabel("Banda larga fixa por 100 hab.")
ax.set_ylabel("Agências por 100k hab.")
save_figure(fig, "03_quadrantes_infra_gap.png")
plt.show()

## 5. Top estados com maior % de municípios sem agência

In [ ]:
uf_sem_agencia = (
    df.groupby("sigla_uf")
    .agg(total=("id_municipio", "count"), sem_agencia=("flag_sem_agencia", "sum"))
    .reset_index()
)
uf_sem_agencia["pct_sem_agencia"] = (
    uf_sem_agencia["sem_agencia"] / uf_sem_agencia["total"] * 100
)
uf_sem_agencia = uf_sem_agencia.sort_values("pct_sem_agencia", ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
uf_sem_agencia.plot(
    x="sigla_uf", y="pct_sem_agencia", kind="barh", ax=ax, color="firebrick"
)
ax.set_title("% de Municípios sem Agência Bancária por UF")
ax.set_xlabel("% sem agência")
save_figure(fig, "03_pct_sem_agencia_uf.png")
plt.show()

In [ ]:
report = {
    "municipios_sem_agencia": int(sem_agencia),
    "pct_municipios_sem_agencia": round(sem_agencia / len(df) * 100, 2),
    "distribuicao_quadrantes": df["quadrante"].value_counts().to_dict(),
    "top_uf_sem_agencia": uf_sem_agencia.tail(5).to_dict(orient="records"),
}
save_json(report, "03_infra_gap_bancario.json")